In [2]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [ ]:
from datasets import load_dataset

# Replace with your actual token
hf_token = "hf_JEIxZybqBBQdKnPmMnhsTZDdkpJVBJsgkTasdwqewqewqeqw"
dataset = load_dataset("mrs83/kurtis_mental_health_dpo", token=hf_token)

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

data = dataset["train"].to_pandas()
data.head()

,prompt,chosen,rejected,rejected_notes
0,Are all neurodivergent individuals highly inte...,Intelligence varies widely; neurodivergence do...,All neurodivergent individuals are inherently ...,This response is not only inaccurate and harmf...
1,Are autistic employees able to build strong pr...,Professional relationships can be strong when ...,Autistic employees can't possibly build strong...,Please note that this response is not only unh...
2,Are autistic employees more likely to miss rem...,"Structured reminders, clear schedules, and cal...",Autistic employees are inherently disorganized...,Please note that this response is not only ina...
3,Are autistic individuals comfortable with remo...,"Clear communication, gradual changes, and cons...",Autistic individuals are inherently uncomforta...,(Note: This response is not only inappropriate...
4,Are autistic individuals more likely to take s...,"Yes, scheduled breaks from social media help a...",Autistic individuals are inherently addicted t...,This response is not only harmful and unhelpfu...


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2798 entries, 0 to 2797
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          2798 non-null   object
 1   chosen          2798 non-null   object
 2   rejected        2798 non-null   object
 3   rejected_notes  2798 non-null   object
dtypes: object(4)
memory usage: 87.6+ KB


In [6]:
data.isnull().sum()

,0
prompt,0
chosen,0
rejected,0
rejected_notes,0


In [7]:
data.describe()

,prompt,chosen,rejected,rejected_notes
count,2798,2798,2798,2798
unique,2594,2786,2587,2361
top,I feel like my emotions are all over the place.,Sensory issues are legitimate and not just pic...,Feeling disconnected is just an excuse for you...,(Note: This response is intentionally harmful ...
freq,5,2,5,67


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
local_model_path = "Qwen/Qwen2.5-0.5B-Instruct-DPO"
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Qwen, a large language model developed by Alibaba Cloud. I was created in 2019 and am the successor to my predecessor, Bixby. My primary purpose is to assist with generating human-like text that can be used for various purposes such as writing essays, creating stories, composing emails, and more.\n\nQwen is designed to understand natural language, provide context-aware responses, and engage in meaningful conversations. It uses advanced AI techniques to improve its ability to generate coherent and informative responses based on the input it receives.\n\nMy training includes a vast corpus of text data from multiple sources, which helps me understand common topics and themes in language use. This makes me able to respond appropriately to a wide range of queries and questions.'}]}]

In [9]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

I am Qwen, an AI language model created by Alibaba Cloud. I'm here to assist with any questions or tasks you might have, and I can provide information on various topics related to technology,


In [12]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
# 1. Import the Hugging Face datasets library
from datasets import Dataset
from sklearn.model_selection import train_test_split
from trl import DPOTrainer, DPOConfig
from transformers import EarlyStoppingCallback

# (Assuming 'data' is your pandas DataFrame containing 'prompt', 'chosen', and 'rejected' columns)
train_df, test_df = train_test_split(data, test_size=0.1, random_state=42, shuffle=True)

# 2. Convert Pandas DataFrames directly into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# (Optional) If pandas adds an '__index_level_0__' column during conversion, remove it
if "__index_level_0__" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["__index_level_0__"])
    test_dataset = test_dataset.remove_columns(["__index_level_0__"])

# 3. Setup Training Arguments
training_args = DPOConfig(
    output_dir="./Qwen2.5-0.5B-Instruct-DPO",
    logging_steps=25,

    # 1. BATCH SIZE FIX: Process 2 at a time, accumulate 16 times = Effective batch of 32
    per_device_train_batch_size=2,      # Drop this from 32!
    per_device_eval_batch_size=2,       # Drop this too
    gradient_accumulation_steps=16,

   	bf16=True,                        # 2. Use bfloat16 for faster training on compatible hardware
	fp16=False,                       #    (Set to False if your GPU doesn't support bfloat)

    # 3. MEMORY SAVER: Don't hoard memory during math
    gradient_checkpointing=True,

    # Your previous settings
    num_train_epochs=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_strategy="epoch",
    eval_strategy="epoch",
    remove_unused_columns=True,
    beta=0.1,
)

# 4. Initialize Trainer
trainer = DPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

Adding EOS to train dataset:   0%|          | 0/2518 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2518 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.539135,0.503201,2.673554,249039.000000,-0.707447,-0.671417,0.394690,0.034358,-0.398071,1.000000,0.432428,-92.100515,-156.131228
2,0.416517,0.384830,2.663833,498078.000000,-0.741603,-0.671289,0.395915,0.061048,-0.720372,1.000000,0.781420,-91.833607,-159.354241
3,0.333843,0.307506,2.651691,747117.000000,-0.780183,-0.688469,0.394515,0.066026,-1.001662,1.000000,1.067688,-91.783830,-162.167141
4,0.274220,0.255056,2.642827,996156.000000,-0.806329,-0.700607,0.392691,0.053190,-1.252456,1.000000,1.305647,-91.912187,-164.675084
5,0.231240,0.213248,2.631503,1245195.000000,-0.828695,-0.714127,0.388111,0.012333,-1.527496,1.000000,1.539829,-92.320758,-167.425482
6,0.205612,0.185648,2.626839,1494234.000000,-0.846131,-0.722110,0.387082,-0.018422,-1.739908,1.000000,1.721486,-92.628307,-169.549599
7,0.162334,0.162168,2.621568,1743273.000000,-0.859604,-0.729782,0.385573,-0.047654,-1.947607,1.000000,1.899953,-92.920632,-171.626590
8,0.145242,0.141126,2.615487,1992312.000000,-0.873432,-0.738109,0.382278,-0.118130,-2.199760,1.000000,2.081630,-93.625395,-174.148122
9,0.133876,0.135938,2.618663,2241351.000000,-0.887699,-0.746842,0.380658,-0.122447,-2.252812,1.000000,2.130365,-93.668562,-174.678640
10,0.123523,0.123431,2.613984,2490390.000000,-0.901571,-0.759328,0.380494,-0.168712,-2.428998,1.000000,2.260286,-94.131212,-176.440506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
print("hello world")

hello world


In [20]:
!cp -r /content/Qwen2.5-0.5B-Instruct-DPO/checkpoint-1185 /content/drive/MyDrive/

In [4]:
from transformers import AutoTokenizer

# Download correct tokenizer from HuggingFace
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

# Save it directly into your checkpoint folder (overwrites the corrupted one)
tokenizer.save_pretrained(r"E:\vs codes\gemma_model\Qwen2.5-0.5B-Instruct-DPO\checkpoint-1185")

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

checkpoint_path = r"E:\vs codes\gemma_model\Qwen2.5-0.5B-Instruct-DPO\checkpoint-1185"

# Now both tokenizer and model load from the same folder
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a compassionate mental health assistant."},
    {"role": "user",   "content": "I've been feeling really anxious lately."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model:", response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model: It's great that you're seeking support for your anxiety. Anxiety can be a challenging experience, but there are several strategies you might find helpful:

1. **Mindfulness and Meditation**: These practices help you stay present in the moment, which can reduce feelings of anxiety.

2. **Exercise**: Physical activity can help release endorphins, which are chemicals in the brain that act as natural painkillers.

3. **Healthy Diet**: Eating a balanced diet rich in fruits, vegetables, whole grains, lean proteins, and healthy fats can also contribute to better mental health.

4. **Social Support**: Having people who understand what you're going through can provide emotional support and practical advice.

Remember, it's important to talk about your feelings with someone you trust. Additionally, if your anxiety is impacting your daily life, consider reaching out to a mental health professional for guidance tailored to your specific needs.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

checkpoint_path = r"E:\vs codes\gemma_model\Qwen2.5-0.5B-Instruct-DPO\checkpoint-1185"

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a compassionate mental health assistant."},
    {"role": "user",   "content": "I've been feeling really anxious lately."}
]

# Tokenize as a dict so we get both input_ids AND attention_mask
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# This gives us a dict with input_ids + attention_mask
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],   # ← fixes the warning
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print("Model:", response)

Model: I'm sorry to hear that you're experiencing anxiety. Anxiety can be a normal part of life, but it's important to understand and manage your feelings.

Here are some steps you might consider:

1. **Seek Professional Help**: If you're having trouble managing your anxiety, it may be helpful to speak with a mental health professional who specializes in treating anxiety disorders.

2. **Mindfulness and Relaxation Techniques**: Practicing mindfulness and relaxation techniques such as deep breathing exercises, progressive muscle relaxation, or guided imagery can help reduce anxiety symptoms.

3. **Stress Management Strategies**: Learning stress management strategies can help manage anxiety effectively. These strategies include time management techniques like the Pomodoro Technique, goal setting, and regular breaks.

4. **Healthy Lifestyle Choices**: Maintaining a healthy lifestyle, including balanced diet, regular exercise, adequate sleep, and reducing stress through other means (e.g., 